# FedBuff with staleness weights (notebook 05)

Buffered async FL that applies **deltas** on top of the current model. Based on Nguyen et al., *Federated Learning with Buffered Asynchronous Aggregation* (2022).

**How it works**

1. Clients train continuously; arriving updates wait in a buffer.
2. Each update gets the weight `samples / (1 + staleness) ^ power`.
3. When the buffer is full: `global += server_rate × weighted average of the deltas`.

> **Note:** unlike notebook 04, each update is applied as a *change* on top of the **current** global model, so stale updates can't erase recent progress.

> **Caution:** weights are normalised inside the buffer, so staleness only decides how the buffered updates are weighted **against each other**. A buffer where every update is stale still moves the model by the full `server_rate`.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FedBuff(Approach):
    name = "fedbuff"
    label = "Async staleness weighted K=5"          # shown in plot legends
    buffer_size = 5
    server_rate = 0.5
    power = 1.0                   # staleness penalty strength inside the buffer

    def start(self, server):
        self.buffer = []
        super().start(server)

    def on_update(self, server, update):
        self.buffer.append(update)
        if len(self.buffer) == self.buffer_size:
            weights = [float(u.samples) / ((1.0 + u.staleness) ** self.power) for u in self.buffer]
            server.set_model(apply_delta(server.model, self.buffer, weights, self.server_rate), self.buffer)
            self.buffer = []

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FedBuff(buffer_size=10))`. It is saved as a separate run.

In [3]:
results = run(FedBuff())

fedbuff                      seed=42   val= 94.14%  acc= 95.20%  time=   560.2s  updates=  100  worst10%= 94.67%  wall=5.7s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
20,42,560.1644,100,0.952,0.1598,0.9467,2.8
